In [1]:
import pickle
import pandas as pd
import os
from env_action.environment  import FJSP_under_uncertainties_Env
from stable_baselines3       import DQN

models_dir = 'models/default_2024-07-21_19-11-22'
model_path = os.path.join(models_dir, f"DQN_.zip")

directory           = 'SMALL'
planning_horizon    = 480*60
critical_machines   = {5, 6, 7, 8, 9, 10, 11, 12, 13, 21, 22, 26, 27}
ReworkProbability   = 0.03
maxtime             = 20
PopSize             = 40
WeibullDistribution = pd.read_excel('DATA/DataMaster.xlsx', sheet_name='Distribution')
K                   = 30
maxJob              = 1320 # for normalization
maxOpe              = 5760 # for normalization


# Default setting
reward_ratio            = 0.99
learning_rate           = 1e-4
explore_fraction_phase1 = 0.4
explore_fraction_phase2 = 0.3
explore_fraction_phase3 = 0.2
num_timestep_phase1     = 3000
num_timestep_phase2     = 12000
num_timestep_phase3     = 25000


purpose = 'default'

if purpose != 'default':
    #------- EXPLORATION FRACTION------------
    if purpose == "SA EF 0.8": # already
        explore_fraction_phase1 = 0.8
        explore_fraction_phase2 = 0.7
        explore_fraction_phase3 = 0.6
    
    if purpose == "SA EF 0.7": # already
        explore_fraction_phase1 = 0.7
        explore_fraction_phase2 = 0.6
        explore_fraction_phase3 = 0.5

    if purpose == "SA EF 0.6": # already
        explore_fraction_phase1 = 0.6
        explore_fraction_phase2 = 0.5
        explore_fraction_phase3 = 0.4

    if purpose == "SA EF 0.5": # already
        explore_fraction_phase1 = 0.5
        explore_fraction_phase2 = 0.4
        explore_fraction_phase3 = 0.3

    #---------- REWARD RATIO-------------
    if purpose == "SA reward 0.90":
        reward_ratio = 0.90
    if purpose == "SA reward 0.80":
        reward_ratio = 0.80
    if purpose == "SA reward 0.70":
        reward_ratio = 0.80
    if purpose == "SA reward 0.60":
        reward_ratio = 0.80


    #---------- CURRICULUM -------------
    if purpose == "SA CL 13-13-14": # already
        num_timestep_phase1     = 13000
        num_timestep_phase2     = 13000
        num_timestep_phase3     = 14000
    if purpose == "SA CL 8-16-16": # already
        num_timestep_phase1     = 8000
        num_timestep_phase2     = 16000
        num_timestep_phase3     = 16000
    if purpose == "SA CL 3-9-28": # already
        num_timestep_phase1     = 3000
        num_timestep_phase2     = 9000
        num_timestep_phase3     = 28000
    if purpose == "SA CL 3-6-31": # already
        num_timestep_phase1     = 3000
        num_timestep_phase2     = 6000
        num_timestep_phase3     = 31000

    #-----------LEARNING RATE ---------
    if purpose == "SA LR 1e-2 ":
        learning_rate = 1e-2
    if purpose == "SA LR 1e-3 ":
        learning_rate = 1e-3
    if purpose == "SA LR 1e-5 ":
        learning_rate = 1e-5
    if purpose == "SA LR 1e-6 ":
        learning_rate = 1e-6

In [ ]:
with open('VALIDATION/SMALL/pickle_valid_instances_480.pkl', 'rb') as f:
    valid_instances = pickle.load(f)
with open('VALIDATION/SMALL/pickle_valid_scenarios_480.pkl', 'rb') as f:
    valid_scenarios = pickle.load(f)

results = []
method = 'predictive-reactive DQN'
InstanceList = [f'valid{i+1}' for i in range(15)]
ScenarioList = ['A', 'B', 'C']

valenv = FJSP_under_uncertainties_Env(False, False, valid_instances, valid_scenarios, K, WeibullDistribution, critical_machines, 
                                      ReworkProbability, planning_horizon, PopSize, maxtime, maxJob, maxOpe, reward_ratio)


model = DQN.load(model_path, env=valenv)

for run_time in range(5):
    print("----------- Run Time", run_time)
    for instance_id in InstanceList:
        print("-----------", instance_id)
        for scenario_id in ScenarioList:
            print("-----", scenario_id)
            # Reset the environment with the new dataset
            # valenv.reset(test=True, 
            #         datatest=instance_id, 
            #         scenariotest=scenario_id)
            
            obs, info = valenv.reset(test=True, 
                    datatest=instance_id, 
                    scenariotest=scenario_id)
            done = False
            
            while not done:
                action, _states = model.predict(obs, deterministic= True)
                obs, reward, done, truncated, info = valenv.step(action)
            
            tardiness = valenv.calc_tardiness()
        
            results.append({'RunTime'   : run_time,
                            'Method'    : method,
                            'InstanceID': instance_id,
                            'ScenarioID': scenario_id,
                            'Tardiness' : tardiness
                            })


In [ ]:
# df = pd.DataFrame(results)
# file_name = f"VALIDATION/{purpose}.xlsx"
# df.to_excel(file_name, index=False)